# GSM8K Data Exploration

Dataset stats, answer distributions, question lengths, reasoning step counts.

In [ ]:
import sys
from typing import List

import matplotlib.pyplot as plt
from datasets import load_dataset

sys.path.insert(0, "..")
from src.evaluate import extract_ground_truth

In [ ]:
ds = load_dataset("gsm8k", "main")
train, test = ds["train"], ds["test"]
print(f"Train: {len(train)}, Test: {len(test)}")

In [ ]:
for i in range(3):
    ex = train[i]
    print(f"=== Example {i} ===")
    print("Q:", ex["question"])
    print("A:", ex["answer"])
    print("Numeric:", extract_ground_truth(ex["answer"]))
    print()

## Answer distribution

In [ ]:
train_ans = [extract_ground_truth(ex["answer"]) for ex in train]
train_ans = [a for a in train_ans if a is not None]
test_ans = [extract_ground_truth(ex["answer"]) for ex in test]
test_ans = [a for a in test_ans if a is not None]

print(f"Valid — train: {len(train_ans)}, test: {len(test_ans)}")
print(f"Min: {min(train_ans)}, Max: {max(train_ans)}, "
      f"Mean: {sum(train_ans)/len(train_ans):.1f}, "
      f"Median: {sorted(train_ans)[len(train_ans)//2]}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.hist([a for a in train_ans if a <= 500], bins=50, edgecolor="black", alpha=0.7)
ax1.set_title("Train answers (0–500)")
ax1.set_xlabel("Answer value")

ax2.hist([a for a in test_ans if a <= 500], bins=50, edgecolor="black",
         alpha=0.7, color="orange")
ax2.set_title("Test answers (0–500)")
ax2.set_xlabel("Answer value")

plt.tight_layout()
plt.show()

## Question length

In [ ]:
q_lens = [len(ex["question"].split()) for ex in train]
print(f"Question length (words): mean={sum(q_lens)/len(q_lens):.1f}, "
      f"min={min(q_lens)}, max={max(q_lens)}")

plt.figure(figsize=(9, 3.5))
plt.hist(q_lens, bins=40, edgecolor="black", alpha=0.7)
plt.title("Question length distribution")
plt.xlabel("Words")
plt.ylabel("Count")
plt.show()

## Reasoning steps per problem

Count non-empty lines before the `####` delimiter as a rough proxy for problem complexity.

In [ ]:
def _count_steps(answer_text: str) -> int:
    reasoning = answer_text.split("####")[0] if "####" in answer_text else answer_text
    return len([l for l in reasoning.strip().split("\n") if l.strip()])

steps = [_count_steps(ex["answer"]) for ex in train]
print(f"Steps: mean={sum(steps)/len(steps):.1f}, min={min(steps)}, max={max(steps)}")

plt.figure(figsize=(9, 3.5))
plt.hist(steps, bins=range(1, max(steps)+1), edgecolor="black", alpha=0.7)
plt.title("Reasoning steps per problem")
plt.xlabel("Steps")
plt.ylabel("Count")
plt.show()